# Multimodal Robotics Data Pipeline: Core OpenCV Implementation
**Assessment**: Jupiter AI Labs - ML Engineer (Robotics)

This notebook provides simple, efficient, and readable implementations for the core OpenCV computer vision tasks:
1. **Video Inspection & Metadata Extraction** (Tasks 1 & 4)
2. **Multi-Camera Synchronization & 2x2 Tiled Video** (Task 2)
3. **21-Point Hand-Keypoint Annotation** (Task 3)
4. **Visual Odometry / SLAM Camera Trajectory Estimation** (Task 5)

In [ ]:
import os
import json
import urllib.request
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Setup input and output directories
EGO_DIR = Path("drive-download-20261015T111339Z-1-001/ego")
UMI_DIR = Path("drive-download-20261015T111339Z-1-001/umi")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("OpenCV Version:", cv2.__version__)
print("MediaPipe Version:", mp.__version__)

## 1. Video Inspection & Technical Properties (Tasks 1 & 4)
Inspect resolution, FPS, frame count, and duration for all Ego and UMI video streams using `cv2.VideoCapture`.

In [ ]:
def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frames / fps if fps > 0 else 0
    cap.release()
    return {
        "file": video_path.name,
        "resolution": f"{width}x{height}",
        "fps": round(fps, 2),
        "frames": frames,
        "duration_s": round(duration, 2)
    }

video_files = sorted(list(EGO_DIR.glob("*.MP4"))) + sorted(list(UMI_DIR.glob("*.MP4")))

print(f"{'File':<25} | {'Resolution':<12} | {'FPS':<6} | {'Frames':<8} | {'Duration (s)':<12}")
print("-" * 75)
for vf in video_files:
    info = get_video_info(vf)
    print(f"{info['file']:<25} | {info['resolution']:<12} | {info['fps']:<6} | {info['frames']:<8} | {info['duration_s']:<12}")

## 2. Multi-Camera Synchronization & 2x2 Tiled Video (Task 2)
Read frames simultaneously from the 4 Ego cameras (`HL`, `HR`, `WL`, `WR`), resize each to 640x360, add camera identifiers, stitch them into a 2x2 synchronized grid (1280x720), and write using `cv2.VideoWriter`.

In [ ]:
ego_paths = sorted(list(EGO_DIR.glob("*.MP4")))
caps = [cv2.VideoCapture(str(p)) for p in ego_paths]
names = [p.stem.split('_')[0] for p in ego_paths]

fps = caps[0].get(cv2.CAP_PROP_FPS)
sync_output_path = OUTPUT_DIR / "ego_synchronized_2x2.mp4"
writer = cv2.VideoWriter(str(sync_output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (1280, 720))

FRAMES_TO_PROCESS = 150
for _ in range(FRAMES_TO_PROCESS):
    frames = []
    for cap, name in zip(caps, names):
        ret, frame = cap.read()
        if not ret:
            break
        resized = cv2.resize(frame, (640, 360))
        cv2.putText(resized, f"Cam: {name}", (25, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
        frames.append(resized)
    
    if len(frames) != 4:
        break
        
    top_row = np.hstack([frames[0], frames[1]])
    bottom_row = np.hstack([frames[2], frames[3]])
    tiled_frame = np.vstack([top_row, bottom_row])
    writer.write(tiled_frame)

for cap in caps:
    cap.release()
writer.release()

print(f"Saved synchronized 2x2 video: {sync_output_path}")

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(tiled_frame, cv2.COLOR_BGR2RGB))
plt.title("Synchronized 2x2 Ego Cameras Preview")
plt.axis("off")
plt.show()

## 3. Ego 21-Point Hand-Keypoint Annotation (Task 3)
Detect hands in the Ego camera video, extract all 21 keypoints $(x, y, z)$ per detected hand, draw the hand skeleton on frames with OpenCV, and save structured annotations to JSON.

In [ ]:
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),        # Index finger
    (5, 9), (9, 10), (10, 11), (11, 12),   # Middle finger
    (9, 13), (13, 14), (14, 15), (15, 16), # Ring finger
    (13, 17), (17, 18), (18, 19), (19, 20), (0, 17) # Pinky & Palm
]

# Auto-download hand_landmarker.task model if missing
model_path = "hand_landmarker.task"
if not os.path.exists(model_path):
    print("Downloading hand_landmarker.task model...")
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
    urllib.request.urlretrieve(url, model_path)

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)

video_path = EGO_DIR / "HL_CH01_005_1013.MP4"
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

annotated_video_path = OUTPUT_DIR / "annotated_hands.mp4"
writer = cv2.VideoWriter(str(annotated_video_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))

annotations = []
MAX_FRAMES = 100

for frame_idx in range(MAX_FRAMES):
    ret, frame = cap.read()
    if not ret:
        break
        
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    result = detector.detect(mp_img)
    
    frame_data = {"frame_idx": frame_idx, "hands": []}
    if result.hand_landmarks:
        for hand_idx, landmarks in enumerate(result.hand_landmarks):
            handedness = result.handedness[hand_idx][0].category_name if result.handedness else "Hand"
            score = float(result.handedness[hand_idx][0].score) if result.handedness else 1.0
            
            pts = []
            for lm in landmarks:
                pts.append({"x": round(lm.x, 4), "y": round(lm.y, 4), "z": round(lm.z, 4)})
                px, py = int(lm.x * W), int(lm.y * H)
                cv2.circle(frame, (px, py), 5, (0, 0, 255), -1)
            
            for start_idx, end_idx in HAND_CONNECTIONS:
                p1 = (int(landmarks[start_idx].x * W), int(landmarks[start_idx].y * H))
                p2 = (int(landmarks[end_idx].x * W), int(landmarks[end_idx].y * H))
                cv2.line(frame, p1, p2, (0, 255, 0), 2)
                
            frame_data["hands"].append({"label": handedness, "score": score, "landmarks": pts})
            
    annotations.append(frame_data)
    writer.write(frame)

cap.release()
writer.release()

json_path = OUTPUT_DIR / "hand_keypoints.json"
with open(json_path, "w") as f:
    json.dump(annotations, f, indent=2)

print(f"Saved annotated video to: {annotated_video_path}")
print(f"Saved 21-point hand annotations to: {json_path}")

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f"Ego Hand Landmark Annotation (Frame {frame_idx})")
plt.axis("off")
plt.show()

## 4. Visual Odometry / SLAM Camera Trajectory Estimation (Task 5)
Estimate relative camera motion and 3D trajectory from monocular Ego video using pure OpenCV:
1. **Corner Feature Detection**: `cv2.goodFeaturesToTrack` (Shi-Tomasi)
2. **Feature Tracking**: `cv2.calcOpticalFlowPyrLK` (Lucas-Kanade)
3. **Motion Estimation**: `cv2.findEssentialMat` (5-point RANSAC) & `cv2.recoverPose`
4. **Trajectory Integration**: Accumulate rotation $R$ and translation $t$ to plot camera path.

In [ ]:
video_path = EGO_DIR / "HL_CH01_005_1013.MP4"
cap = cv2.VideoCapture(str(video_path))

W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Camera Intrinsic Matrix (Pinhole model approximation: focal length ~ 1000, center at W/2, H/2)
K = np.array([
    [1000.0,    0.0, W / 2.0],
    [   0.0, 1000.0, H / 2.0],
    [   0.0,    0.0,     1.0]
], dtype=np.float64)

cur_R = np.eye(3)
cur_t = np.zeros((3, 1))
trajectory = [cur_t.flatten()]

ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
prev_pts = cv2.goodFeaturesToTrack(prev_gray, maxCorners=500, qualityLevel=0.01, minDistance=15)

MAX_FRAMES = 150
for _ in range(MAX_FRAMES):
    ret, curr_frame = cap.read()
    if not ret:
        break
        
    curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
    
    curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_pts, None)
    good_prev = prev_pts[status == 1]
    good_curr = curr_pts[status == 1]
    
    if len(good_prev) >= 8:
        E, _ = cv2.findEssentialMat(good_curr, good_prev, K, method=cv2.RANSAC, prob=0.999, threshold=1.0)
        _, R, t, _ = cv2.recoverPose(E, good_curr, good_prev, K)
        
        cur_t = cur_t + cur_R.dot(t)
        cur_R = cur_R.dot(R)
        trajectory.append(cur_t.flatten())
    
    if len(good_curr) < 200:
        prev_pts = cv2.goodFeaturesToTrack(curr_gray, maxCorners=500, qualityLevel=0.01, minDistance=15)
    else:
        prev_pts = good_curr.reshape(-1, 1, 2)
        
    prev_gray = curr_gray

cap.release()
trajectory = np.array(trajectory)

traj_csv = OUTPUT_DIR / "camera_trajectory.csv"
np.savetxt(traj_csv, trajectory, delimiter=",", header="x,y,z", comments="")
print(f"Saved camera trajectory ({len(trajectory)} poses) to: {traj_csv}")

plt.figure(figsize=(8, 6))
plt.plot(trajectory[:, 0], trajectory[:, 2], '-o', color='crimson', markersize=3, label='Estimated Path')
plt.scatter(trajectory[0, 0], trajectory[0, 2], color='green', s=100, label='Start')
plt.scatter(trajectory[-1, 0], trajectory[-1, 2], color='red', s=100, label='End')
plt.title("Visual Odometry: Estimated Camera Trajectory (X vs Z Motion)")
plt.xlabel("X Position (Relative Scale)")
plt.ylabel("Z Position / Depth (Relative Scale)")
plt.grid(True)
plt.legend()
plt.show()